# **SpaceX Falcon 9 First Stage Landing Prediction**
## Lab 3 : Interactive Visual Analytics with Folium

Dans ce notebook, on explore les emplacements des sites de lancement SpaceX sur une carte interactive avec **Folium**, pour répondre à des questions comme :
- Les sites sont-ils proches de l'équateur, de la côte, des voies ferrées, des routes ?
- Sont-ils éloignés des villes ?
- Quels sites ont les meilleurs taux de succès ?

**Tâches** : (1) marquer tous les sites, (2) marquer les lancements réussis/échoués par site, (3) calculer les distances entre un site et ses points d'intérêt proches.

In [1]:
# Fonctionne dans Jupyter / Colab / JupyterLab
%pip install -q folium pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

## Chargement des données

In [3]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
spacex_df = pd.read_csv(URL)
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
print(spacex_df.shape)
spacex_df.head()

(56, 4)


,Launch Site,Lat,Long,class
0,CCAFS LC-40,28.562302,-80.577356,0
1,CCAFS LC-40,28.562302,-80.577356,0
2,CCAFS LC-40,28.562302,-80.577356,0
3,CCAFS LC-40,28.562302,-80.577356,0
4,CCAFS LC-40,28.562302,-80.577356,0


In [4]:
# Une ligne par site de lancement
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


## Task 1 : marquer tous les sites de lancement sur une carte
Pour chaque site : un cercle (`folium.Circle`) et une étiquette avec son nom (`folium.Marker` + `DivIcon`). La carte est centrée sur le Johnson Space Center de la NASA (Houston).

In [5]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Cercle + etiquette pour le NASA Johnson Space Center
folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center')).add_to(site_map)
folium.map.Marker(nasa_coordinate, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
    html='<div style="font-size: 12; color:#d35400;"><b>NASA JSC</b></div>')).add_to(site_map)

# Un cercle et une etiquette par site de lancement
for _, row in launch_sites_df.iterrows():
    coord = [row['Lat'], row['Long']]
    folium.Circle(coord, radius=1000, color='#000000', fill=True).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#000000;"><b>%s</b></div>' % row['Launch Site'])).add_to(site_map)

site_map

**Observations (à confirmer sur ta carte)** : trois sites se trouvent en Floride (CCAFS LC-40, CCAFS SLC-40, KSC LC-39A) et un en Californie (VAFB SLC-4E). Tous sont situés **près de la côte** et à **faible latitude** (proches de l'équateur, ce qui donne un gain de vitesse de rotation de la Terre).

## Task 2 : marquer les lancements réussis et échoués pour chaque site
On regroupe les marqueurs avec `MarkerCluster` : **vert** = succès (`class = 1`), **rouge** = échec (`class = 0`).

In [6]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')

site_map2 = folium.Map(location=nasa_coordinate, zoom_start=5)
marker_cluster = MarkerCluster().add_to(site_map2)

for _, row in spacex_df.iterrows():
    folium.Marker([row['Lat'], row['Long']],
                  icon=folium.Icon(color='white', icon_color=row['marker_color']),
                  popup=f"{row['Launch Site']} - {'Success' if row['class'] == 1 else 'Failure'}").add_to(marker_cluster)

site_map2

In [7]:
# Taux de succes par site (utile pour interpreter la carte)
success_by_site = spacex_df.groupby('Launch Site')['class'].agg(['sum', 'count', 'mean'])
success_by_site.columns = ['Successes', 'Launches', 'Success rate']
success_by_site.sort_values('Success rate', ascending=False)

,Successes,Launches,Success rate
Launch Site,,,
KSC LC-39A,10,13,0.769231
CCAFS SLC-40,3,7,0.428571
VAFB SLC-4E,4,10,0.400000
CCAFS LC-40,7,26,0.269231


**Observations (à confirmer avec le tableau ci-dessus)** : le site avec le meilleur taux de succès est celui qui affiche le plus de marqueurs verts ; compare chaque site à l'aide de la colonne `Success rate`.

## Task 3 : distances entre un site de lancement et ses points d'intérêt
On utilise le plugin `MousePosition` pour lire les coordonnées en survolant la carte, puis la formule de Haversine pour calculer les distances en km.

In [8]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """Distance en km entre deux points GPS (formule de Haversine)."""
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

In [9]:
# Carte centree sur un site avec affichage des coordonnees de la souris
site = launch_sites_df[launch_sites_df['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
site_lat, site_lon = site['Lat'], site['Long']

map3 = folium.Map(location=[site_lat, site_lon], zoom_start=14)
MousePosition(position='topright', separator=' Long: ', empty_string='NaN', lng_first=False,
              num_digits=20, prefix='Lat:',
              lat_formatter="function(num) {return L.Util.formatNum(num, 5);};",
              lng_formatter="function(num) {return L.Util.formatNum(num, 5);};").add_to(map3)
folium.Marker([site_lat, site_lon], popup='CCAFS SLC-40').add_to(map3)
map3

**Important** : survole la carte et relève, pour chaque point d'intérêt, ses coordonnées exactes (côte, voie ferrée, route, ville). Remplace les valeurs ci-dessous. **Les valeurs fournies sont seulement approximatives** pour que le notebook s'exécute, elles doivent être remplacées par tes relevés.

In [10]:
# A REMPLACER par tes releves avec MousePosition (valeurs approximatives)
points_of_interest = {
    'Coastline': (28.5636, -80.5680),
    'Railway':   (28.5721, -80.5853),
    'Highway':   (28.5633, -80.5709),
    'City (Cape Canaveral)': (28.4058, -80.6048),
}

In [11]:
def draw_distance(m, site_lat, site_lon, poi_lat, poi_lon, label):
    d = calculate_distance(site_lat, site_lon, poi_lat, poi_lon)
    folium.Marker([poi_lat, poi_lon], icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s: %.2f KM</b></div>' % (label, d))).add_to(m)
    folium.PolyLine([[site_lat, site_lon], [poi_lat, poi_lon]], weight=1).add_to(m)
    return d

distances = {}
for label, (lat, lon) in points_of_interest.items():
    distances[label] = draw_distance(map3, site_lat, site_lon, lat, lon, label)

pd.DataFrame({'Point of interest': list(distances.keys()), 'Distance (km)': [round(v, 2) for v in distances.values()]})

,Point of interest,Distance (km)
0,Coastline,0.86
1,Railway,1.29
2,Highway,0.58
3,City (Cape Canaveral),17.72


In [12]:
map3

In [13]:
# Sauvegarde des cartes en HTML (pour les ouvrir dans un navigateur ou prendre des captures)
site_map.save('launch_sites_map.html')
site_map2.save('launch_outcomes_map.html')
map3.save('proximity_map.html')